In [5]:
"""
Download household-survey indicators from the ECLAC / CEPALSTAT open-data API.

Setup:  pip install requests pandas

Notebook / Interactive window (run in separate cells):
    run_download(list_only=True)   # just list matching indicators
    run_download()                 # list + download data (default)
    reprocess_raw()                # rebuild CSVs from saved JSON (no re-download)
    inspect_raw()                  # print the structure of one raw response
    build_database()               # put all indicator CSVs into one SQLite file

Terminal:
    python cepalstat_households.py --list-only
    python cepalstat_households.py --keywords household hogares --lang en
"""

import argparse
import json
import re
import sqlite3
import sys
import time
from pathlib import Path

import pandas as pd
import requests

BASE = "https://api-cepalstat.cepal.org/cepalstat/api/v1"

# >>> CHANGE THIS if you want a specific location <<<
OUT_DIR = Path.home() / "Documents" / "cepalstat_households"

DEFAULT_KEYWORDS = ["household", "households", "hogar", "hogares", "household survey"]


# ---------------------------------------------------------------- helpers
def get_json(url, params=None, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            if attempt == retries - 1:
                raise
            print(f"  retry {attempt + 1} after error: {e}")
            time.sleep(2 * (attempt + 1))


def walk_tree(node, path=None):
    """Recursively yield every indicator node with its thematic path."""
    path = path or []
    if isinstance(node, list):
        for n in node:
            yield from walk_tree(n, path)
    elif isinstance(node, dict):
        name = node.get("name") or node.get("label") or ""
        here = path + [name] if name else path
        children = node.get("children")
        is_indicator = (
            str(node.get("type", "")).lower() == "indicator"
            or (node.get("indicator_id") is not None and not children)
        )
        if is_indicator:
            yield {
                "indicator_id": node.get("indicator_id", node.get("id")),
                "name": name,
                "path": " > ".join(here[:-1]),
            }
        if children:
            yield from walk_tree(children, here)
        for key in ("body", "data"):
            if key in node:
                yield from walk_tree(node[key], path)


def find_indicators(lang, keywords):
    tree = get_json(f"{BASE}/thematic-tree", {"lang": lang, "format": "json"})
    all_ind = pd.DataFrame(list(walk_tree(tree)))
    if all_ind.empty:
        raise RuntimeError(
            "No indicators found in the thematic tree. The response structure may "
            "differ from what the parser expects; inspect the raw JSON."
        )
    all_ind = all_ind.drop_duplicates("indicator_id")
    print(f"Thematic tree contains {len(all_ind)} indicators in total.")
    all_ind.to_csv(OUT_DIR / "all_indicators.csv", index=False, encoding="utf-8-sig")

    pattern = re.compile("|".join(re.escape(k) for k in keywords), re.IGNORECASE)
    text = all_ind["name"].fillna("") + " " + all_ind["path"].fillna("")
    return all_ind[text.str.contains(pattern)].reset_index(drop=True)


def _find_members(obj):
    """Return {id: label} from any list of dicts that carry an id and a name/label."""
    out = {}
    if isinstance(obj, list):
        for m in obj:
            if isinstance(m, dict) and "id" in m:
                label = m.get("name") or m.get("label") or m.get("description")
                if label is not None:
                    out[m["id"]] = label
    return out


def flatten(payload):
    """Turn a CEPALSTAT indicator JSON response into a DataFrame with readable labels."""
    body = payload.get("body", payload) if isinstance(payload, dict) else {}
    data = pd.DataFrame(body.get("data", []))
    if data.empty:
        return data

    dims = body.get("dimensions", [])
    if isinstance(dims, dict):
        dims = list(dims.values())

    for dim in dims:
        if not isinstance(dim, dict):
            continue
        col = f"dim_{dim.get('id')}"
        if col not in data.columns:
            continue
        mapping = {}

In [6]:
matches = run_download()  

Saving files to: /Users/twentysixconsulting/Documents/cepalstat_households

Thematic tree contains 1739 indicators in total.
109 indicators match ['household', 'households', 'hogar', 'hogares', 'household survey']

     indicator_id                                                                   name                                                                   path
0            2520  Number of women according to maternity by relationship with the he...  General subject classification > Demographic and social > Demograp...
1            2584  Relative distribution of women according to maternity by relations...  General subject classification > Demographic and social > Demograp...
2            2597  Percentage of mothers according to maternity by relationship with ...  General subject classification > Demographic and social > Demograp...
3            2916  Number of women according to maternity by relationship with the he...  General subject classification > Demographic and social

In [8]:
import json
import pandas as pd
from pathlib import Path

folder = Path.home() / "Documents" / "cepalstat_households"


def _find_members(obj):
    out = {}
    if isinstance(obj, list):
        for m in obj:
            if isinstance(m, dict) and "id" in m:
                label = m.get("name") or m.get("label") or m.get("description")
                if label is not None:
                    out[m["id"]] = label
    return out


def flatten(payload):
    body = payload.get("body", payload) if isinstance(payload, dict) else {}
    data = pd.DataFrame(body.get("data", []))
    if data.empty:
        return data

    dims = body.get("dimensions", [])
    if isinstance(dims, dict):
        dims = list(dims.values())

    for dim in dims:
        if not isinstance(dim, dict):
            continue
        col = f"dim_{dim.get('id')}"
        if col not in data.columns:
            continue
        mapping = {}
        for val in dim.values():
            if isinstance(val, list):
                mapping.update(_find_members(val))
        if mapping:
            label = dim.get("name") or dim.get("label") or col
            data[label] = data[col].map(mapping)
            if data[label].isna().all():
                data[label] = data[col].astype(str).map(
                    {str(k): v for k, v in mapping.items()}
                )
    return data


# Rebuild every CSV from the saved JSON (no download)
for f in sorted((folder / "raw").glob("*.json")):
    df = flatten(json.load(open(f, encoding="utf-8")))
    df.to_csv(folder / f"indicator_{f.stem}.csv", index=False, encoding="utf-8-sig")

# Check one file
df = pd.read_csv(folder / "indicator_118.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(10074, 12)
['value', 'source_id', 'notes_ids', 'iso3', 'dim_334', 'dim_326', 'dim_208', 'dim_29117', 'Country__ESTANDAR', 'Persons', 'Years__ESTANDAR', 'Geographical area']


,value,source_id,notes_ids,iso3,dim_334,dim_326,dim_208,dim_29117,Country__ESTANDAR,Persons,Years__ESTANDAR,Geographical area
0,2.1,26,8620,NaN,335,327,43053,29170,Latin America (simple average),1 person,2000,National
1,6.5,26,8620,NaN,344,327,43053,29170,Latin America (simple average),10 persons and over,2000,National
2,6.7,26,8620,NaN,336,327,43053,29170,Latin America (simple average),2 persons,2000,National
3,12.7,26,8620,NaN,337,327,43053,29170,Latin America (simple average),3 persons,2000,National
4,19.6,26,8620,NaN,338,327,43053,29170,Latin America (simple average),4 persons,2000,National


In [9]:
df = pd.read_csv(folder / "indicator_118.csv")

df = df.rename(columns={"Country__ESTANDAR": "Country", "Years__ESTANDAR": "Year"})
df = df.drop(columns=[c for c in df.columns if c.startswith("dim_")] + ["source_id", "notes_ids"],
             errors="ignore")

print(df["Country"].unique())            # see aggregates and countries
countries_only = df[df["iso3"].notna()]  # aggregates have no iso3 code
countries_only.head(10)

<ArrowStringArray>
[    'Latin America (simple average)',                          'Argentina',
   'Bolivia (Plurinational State of)',                             'Brazil',
                              'Chile',                           'Colombia',
                         'Costa Rica',                            'Ecuador',
                        'El Salvador',                          'Guatemala',
                           'Honduras',                             'Mexico',
                          'Nicaragua',                             'Panama',
                           'Paraguay',                               'Peru',
                 'Dominican Republic',                            'Uruguay',
 'Venezuela (Bolivarian Republic of)']
Length: 19, dtype: str


,value,iso3,Country,Persons,Year,Geographical area
750,4.200000,ARG,Argentina,1 person,2000,Urban
751,3.300000,ARG,Argentina,10 persons and over,2000,Urban
752,12.200000,ARG,Argentina,2 persons,2000,Urban
753,16.299999,ARG,Argentina,3 persons,2000,Urban
754,22.100000,ARG,Argentina,4 persons,2000,Urban
755,18.299999,ARG,Argentina,5 persons,2000,Urban
756,10.700000,ARG,Argentina,6 persons,2000,Urban
757,6.800000,ARG,Argentina,7 persons,2000,Urban
758,4.300000,ARG,Argentina,8 persons,2000,Urban
759,1.800000,ARG,Argentina,9 persons,2000,Urban


In [11]:
df.shape

(10074, 6)

In [12]:
import json, time, sqlite3, requests
import pandas as pd
from pathlib import Path

BASE = "https://api-cepalstat.cepal.org/cepalstat/api/v1"
folder = Path.home() / "Documents" / "cepalstat_households"
(folder / "raw").mkdir(exist_ok=True)

all_ind = pd.read_csv(folder / "all_indicators.csv")
names = all_ind.set_index("indicator_id")["name"].to_dict()
print("Indicators in tree:", len(all_ind))


def fetch(ind_id, lang="en", retries=3):
    for a in range(retries):
        try:
            r = requests.get(f"{BASE}/indicator/{ind_id}/data",
                             params={"lang": lang, "format": "json"}, timeout=120)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if a == retries - 1:
                raise
            time.sleep(2 * (a + 1))


# ---------- 1) Download everything (resumable: skips files already saved)
failed = []
for i, ind_id in enumerate(all_ind["indicator_id"], 1):
    path = folder / "raw" / f"{ind_id}.json"
    if path.exists():
        continue
    try:
        payload = fetch(ind_id)
        json.dump(payload, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    except Exception as e:
        failed.append((ind_id, str(e)))
    if i % 25 == 0:
        print(f"{i}/{len(all_ind)} processed, {len(failed)} failed")
    time.sleep(0.3)

pd.DataFrame(failed, columns=["indicator_id", "error"]).to_csv(folder / "failed.csv", index=False)
print("Download finished. Failed:", len(failed))


# ---------- 2) Merge everything into one SQLite database
META = {"value", "source_id", "notes_ids", "iso3"}

def standardize(df, ind_id):
    labels = [c for c in df.columns if c not in META and not c.startswith("dim_")]
    country = next((c for c in labels if "country" in c.lower() or "país" in c.lower() or "pais" in c.lower()), None)
    year = next((c for c in labels if "year" in c.lower() or "año" in c.lower() or "ano" in c.lower()), None)
    extra = [c for c in labels if c not in (country, year)]

    out = pd.DataFrame({
        "indicator_id": ind_id,
        "indicator": names.get(ind_id),
        "country": df[country] if country else None,
        "iso3": df["iso3"] if "iso3" in df else None,
        "year": df[year] if year else None,
        "value": df["value"] if "value" in df else None,
    })
    if extra:
        out["breakdown"] = df[extra].astype(str).apply(
            lambda r: " | ".join(f"{c}={v}" for c, v in zip(extra, r)), axis=1)
    else:
        out["breakdown"] = ""
    return out


db_path = folder / "cepalstat_all.db"
if db_path.exists():
    db_path.unlink()
con = sqlite3.connect(db_path)

for n, f in enumerate(sorted((folder / "raw").glob("*.json")), 1):
    ind_id = int(f.stem)
    df = flatten(json.load(open(f, encoding="utf-8")))
    if df is None or df.empty:
        continue
    standardize(df, ind_id).to_sql("data", con, if_exists="append", index=False)
    if n % 100 == 0:
        print(f"merged {n} files")

con.execute("CREATE INDEX idx_country ON data(country)")
con.execute("CREATE INDEX idx_ind ON data(indicator_id)")
con.commit()
print("Rows in database:", con.execute("SELECT COUNT(*) FROM data").fetchone()[0])
print("Saved:", db_path)

Indicators in tree: 1739
25/1739 processed, 0 failed
50/1739 processed, 0 failed
75/1739 processed, 0 failed
100/1739 processed, 0 failed
125/1739 processed, 0 failed
150/1739 processed, 0 failed
175/1739 processed, 0 failed
200/1739 processed, 0 failed
225/1739 processed, 0 failed
275/1739 processed, 0 failed
300/1739 processed, 0 failed
325/1739 processed, 0 failed
350/1739 processed, 0 failed
375/1739 processed, 0 failed
400/1739 processed, 0 failed
425/1739 processed, 0 failed
450/1739 processed, 0 failed
475/1739 processed, 0 failed
500/1739 processed, 0 failed
525/1739 processed, 0 failed
550/1739 processed, 0 failed
575/1739 processed, 0 failed
600/1739 processed, 0 failed
625/1739 processed, 0 failed
650/1739 processed, 0 failed
675/1739 processed, 0 failed
725/1739 processed, 0 failed
750/1739 processed, 0 failed
775/1739 processed, 0 failed
825/1739 processed, 0 failed
850/1739 processed, 0 failed
875/1739 processed, 0 failed
900/1739 processed, 0 failed
925/1739 processed, 0